# Google Colab Host for EduMind AI Backend & Ollama

Run this single cell to clone the repo, install/start local Ollama, pull the LLM model, configure `.env` to point to localhost Ollama, and host the FastAPI backend server publicly using Cloudflare Tunnel.

In [ ]:
# 1. Clone the repository and switch directory
import os
import subprocess
import time
import re

if not os.path.exists('demo-'):
    print("Cloning repository...")
    !git clone -b commitee-process-and-fixes https://github.com/jaynishthakar/demo-.git
    %cd demo-
else:
    print("Repository already exists. Switched to demo- directory and pulling updates...")
    %cd demo-
    !git pull

# 2. Install system dependencies & Ollama
print("Installing zstd and Ollama...")
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

print("Starting Ollama service in the background...")
ollama_log = open("ollama_server.log", "w")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=ollama_log
)
time.sleep(5)  # Allow server to start

# Pull the model
print("Pulling qwen2.5:7b model (this takes ~1-2 mins)...")
!ollama pull qwen2.5:7b

# 3. Install Python dependencies
print("Installing requirements...")
!pip install -r requirements.txt
!pip install nest-asyncio

# 4. Create or write .env file pointing to local Ollama
print("Configuring .env file...")
env_content = """OLLAMA_BASE_URL=http://localhost:11434
NGROK_AUTHTOKEN=3FzpiDTu3BDjqq3oaOwhHZ210R8_2Gek4AzFydoBZwZA3GT6Z
OLLAMA_MODEL=qwen2.5:7b
LLM_BACKEND=ollama
HF_MODEL=Qwen/Qwen2.5-32B-Instruct
HF_TOKEN=
"""

with open(".env", "w") as f:
    f.write(env_content)
print(".env file configured successfully.")

# 5. Start the FastAPI backend server in the background
print("Starting FastAPI backend...")
with open("backend.log", "w") as log_file:
    backend_process = subprocess.Popen(
        ["uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", "8000"],
        stdout=log_file,
        stderr=log_file
)

time.sleep(5)  # Wait for startup

# Check if the process is still running
if backend_process.poll() is None:
    print(f"FastAPI backend started successfully in the background (PID: {backend_process.pid}).")
else:
    print("Backend failed to start. Printing backend.log:")
    with open("backend.log", "r") as log_file:
        print(log_file.read())

# 6. Download and start Cloudflare Tunnel (TryCloudflare) to expose FastAPI
if not os.path.exists("cloudflared"):
    print("Downloading Cloudflare Tunnel client (cloudflared)...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

print("Starting Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Waiting for tunnel URL...")
url_found = False
try:
    for line in iter(tunnel_process.stdout.readline, ""):
        if "trycloudflare.com" in line and not url_found:
            urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if urls:
                print("\n" + "="*60)
                print("YOUR BACKEND SERVER IS HOSTED PUBLICLY AT:")
                print(urls[0])
                print("="*60 + "\n")
                url_found = True
        print(line, end="")
except KeyboardInterrupt:
    print("\nStopping tunnel, backend, and Ollama...")
finally:
    tunnel_process.terminate()
    backend_process.terminate()
    ollama_proc.terminate()
    print("Cleanup complete.")